# Week 5: BioPython Projects - Solutions

Solutions for BioPython, ORF finding, and GC landscape projects.

## Complicated moments explained
These are common points where learners usually get stuck:
- Focus on why each step exists, not only the final answer.
- Compare intermediate outputs to your own attempt, not just final metrics.
- Small implementation differences can still be correct if assumptions match.


In [ ]:
# Install BioPython if needed
# !pip install biopython

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils import gc_fraction
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import pandas as pd

## Part 1: BioPython Basics

### Problem 1.1: FASTA File Processing

In [ ]:
def analyze_fasta(filepath):
    """Analyze sequences in a FASTA file using BioPython."""
    results = []
    
    for record in SeqIO.parse(filepath, 'fasta'):
        seq = str(record.seq)
        gc = gc_fraction(record.seq) * 100
        
        results.append({
            'id': record.id,
            'description': record.description,
            'length': len(seq),
            'gc_content': round(gc, 2),
            'a_count': seq.count('A'),
            't_count': seq.count('T'),
            'g_count': seq.count('G'),
            'c_count': seq.count('C')
        })
    
    return pd.DataFrame(results)

# Example usage (if file exists):
# df = analyze_fasta('sequences.fasta')
# print(df)

### Problem 1.2: Sequence Manipulation

In [ ]:
def sequence_operations(dna_seq):
    """Demonstrate BioPython sequence operations."""
    seq = Seq(dna_seq)
    
    results = {
        'original': str(seq),
        'complement': str(seq.complement()),
        'reverse_complement': str(seq.reverse_complement()),
        'transcribed': str(seq.transcribe()),
        'gc_content': gc_fraction(seq) * 100
    }
    
    # Translate if length is multiple of 3
    if len(seq) % 3 == 0:
        results['translated'] = str(seq.translate())
        results['translated_to_stop'] = str(seq.translate(to_stop=True))
    
    return results

# Test
test_seq = "ATGGCCATGGCGCCCAGAACUGAGAUCAAUAGUACCCGUAUUAACGGGTGA".replace('U', 'T')
ops = sequence_operations(test_seq)
for key, value in ops.items():
    print(f"{key}: {value}")

## Part 2: ORF Finder Class

In [ ]:
class ORFFinder:
    """Complete ORF finder with codon analysis."""
    
    CODON_TABLE = {
        'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
        'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
        'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
        'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
        'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
        'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
        'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
        'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
        'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
        'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
        'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
        'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
        'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
        'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
        'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
        'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
    }
    
    STOP_CODONS = {'TAA', 'TAG', 'TGA'}
    START_CODON = 'ATG'
    
    def __init__(self, sequence, min_length=100):
        """Initialize with a DNA sequence."""
        self.sequence = sequence.upper()
        self.min_length = min_length
        self.orfs = []
    
    def _reverse_complement(self, seq):
        """Get reverse complement of sequence."""
        complement = str.maketrans('ATGC', 'TACG')
        return seq.translate(complement)[::-1]
    
    def _translate(self, dna):
        """Translate DNA to protein."""
        protein = []
        for i in range(0, len(dna) - 2, 3):
            codon = dna[i:i+3]
            aa = self.CODON_TABLE.get(codon, 'X')
            if aa == '*':
                break
            protein.append(aa)
        return ''.join(protein)
    
    def _find_orfs_in_frame(self, sequence, frame, strand):
        """Find ORFs in a specific reading frame."""
        orfs = []
        i = frame
        
        while i < len(sequence) - 2:
            # Look for start codon
            if sequence[i:i+3] == self.START_CODON:
                # Look for stop codon
                for j in range(i + 3, len(sequence) - 2, 3):
                    codon = sequence[j:j+3]
                    if codon in self.STOP_CODONS:
                        orf_seq = sequence[i:j+3]
                        if len(orf_seq) >= self.min_length:
                            # Calculate position in original sequence
                            if strand == '+':
                                start_pos = i
                                end_pos = j + 3
                            else:
                                start_pos = len(self.sequence) - j - 3
                                end_pos = len(self.sequence) - i
                            
                            orfs.append({
                                'start': start_pos,
                                'end': end_pos,
                                'length': len(orf_seq),
                                'frame': frame + 1,
                                'strand': strand,
                                'dna_sequence': orf_seq,
                                'protein': self._translate(orf_seq)
                            })
                        break
            i += 3
        
        return orfs
    
    def find_all_orfs(self):
        """Find ORFs in all 6 reading frames."""
        self.orfs = []
        
        # Forward strand
        for frame in range(3):
            self.orfs.extend(self._find_orfs_in_frame(self.sequence, frame, '+'))
        
        # Reverse strand
        rev_comp = self._reverse_complement(self.sequence)
        for frame in range(3):
            self.orfs.extend(self._find_orfs_in_frame(rev_comp, frame, '-'))
        
        # Sort by length
        self.orfs.sort(key=lambda x: x['length'], reverse=True)
        return self.orfs
    
    def codon_usage(self):
        """Analyze codon usage in all found ORFs."""
        if not self.orfs:
            self.find_all_orfs()
        
        all_codons = []
        for orf in self.orfs:
            dna = orf['dna_sequence']
            codons = [dna[i:i+3] for i in range(0, len(dna) - 2, 3)]
            all_codons.extend(codons)
        
        codon_counts = Counter(all_codons)
        total = sum(codon_counts.values())
        
        usage = {}
        for codon, count in codon_counts.items():
            aa = self.CODON_TABLE.get(codon, 'X')
            usage[codon] = {
                'amino_acid': aa,
                'count': count,
                'frequency': count / total * 100
            }
        
        return usage
    
    def summary(self):
        """Get summary statistics."""
        if not self.orfs:
            self.find_all_orfs()
        
        if not self.orfs:
            return {'total_orfs': 0}
        
        lengths = [orf['length'] for orf in self.orfs]
        forward = sum(1 for orf in self.orfs if orf['strand'] == '+')
        reverse = sum(1 for orf in self.orfs if orf['strand'] == '-')
        
        return {
            'total_orfs': len(self.orfs),
            'forward_strand': forward,
            'reverse_strand': reverse,
            'min_length': min(lengths),
            'max_length': max(lengths),
            'mean_length': sum(lengths) / len(lengths),
            'total_coding_bp': sum(lengths)
        }
    
    def to_dataframe(self):
        """Convert ORFs to pandas DataFrame."""
        if not self.orfs:
            self.find_all_orfs()
        return pd.DataFrame(self.orfs)
    
    def plot_orf_lengths(self):
        """Plot histogram of ORF lengths."""
        if not self.orfs:
            self.find_all_orfs()
        
        lengths = [orf['length'] for orf in self.orfs]
        
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(lengths, bins=30, edgecolor='white', color='steelblue')
        ax.axvline(np.mean(lengths), color='red', linestyle='--', 
                  label=f'Mean: {np.mean(lengths):.0f} bp')
        ax.set_xlabel('ORF Length (bp)')
        ax.set_ylabel('Count')
        ax.set_title('Distribution of ORF Lengths')
        ax.legend()
        return fig


# Test
np.random.seed(42)
test_seq = ''.join(np.random.choice(['A', 'T', 'G', 'C'], size=10000))

finder = ORFFinder(test_seq, min_length=90)
orfs = finder.find_all_orfs()

print("Summary:")
for key, value in finder.summary().items():
    print(f"  {key}: {value}")

print(f"\nTop 5 ORFs:")
df = finder.to_dataframe()
print(df[['start', 'end', 'length', 'strand', 'frame']].head())

finder.plot_orf_lengths()
plt.show()

## Part 3: GC Landscape Class

In [ ]:
class GCLandscape:
    """GC content landscape analyzer and visualizer."""
    
    def __init__(self, sequence, window_size=100, step=10):
        """Initialize with sequence and window parameters."""
        self.sequence = sequence.upper()
        self.window_size = window_size
        self.step = step
        self.positions = None
        self.gc_values = None
    
    def calculate(self):
        """Calculate GC content in sliding windows."""
        # Convert to binary (G/C = 1, A/T = 0)
        gc_binary = np.array([1 if n in 'GC' else 0 for n in self.sequence])
        
        # Use cumsum for efficiency
        cumsum = np.concatenate([[0], np.cumsum(gc_binary)])
        
        # Calculate positions
        self.positions = np.arange(0, len(self.sequence) - self.window_size + 1, self.step)
        ends = self.positions + self.window_size
        
        # Calculate GC values
        self.gc_values = (cumsum[ends] - cumsum[self.positions]) / self.window_size * 100
        
        return self.positions, self.gc_values
    
    def find_gc_islands(self, threshold=60, min_length=200):
        """Find GC-rich regions (potential CpG islands)."""
        if self.gc_values is None:
            self.calculate()
        
        islands = []
        in_island = False
        island_start = 0
        
        for i, (pos, gc) in enumerate(zip(self.positions, self.gc_values)):
            if gc >= threshold and not in_island:
                in_island = True
                island_start = pos
            elif gc < threshold and in_island:
                in_island = False
                island_end = pos
                if island_end - island_start >= min_length:
                    islands.append({
                        'start': island_start,
                        'end': island_end,
                        'length': island_end - island_start,
                        'mean_gc': np.mean(self.gc_values[
                            (self.positions >= island_start) & (self.positions < island_end)
                        ])
                    })
        
        # Check if still in island at end
        if in_island:
            island_end = self.positions[-1] + self.window_size
            if island_end - island_start >= min_length:
                islands.append({
                    'start': island_start,
                    'end': island_end,
                    'length': island_end - island_start,
                    'mean_gc': np.mean(self.gc_values[self.positions >= island_start])
                })
        
        return islands
    
    def statistics(self):
        """Calculate GC statistics."""
        if self.gc_values is None:
            self.calculate()
        
        return {
            'overall_gc': (self.sequence.count('G') + self.sequence.count('C')) / len(self.sequence) * 100,
            'mean_window_gc': np.mean(self.gc_values),
            'std_window_gc': np.std(self.gc_values),
            'min_gc': np.min(self.gc_values),
            'max_gc': np.max(self.gc_values),
            'sequence_length': len(self.sequence),
            'n_windows': len(self.gc_values)
        }
    
    def plot(self, highlight_islands=True, threshold=60):
        """Create GC landscape plot."""
        if self.gc_values is None:
            self.calculate()
        
        fig, ax = plt.subplots(figsize=(14, 5))
        
        # Main GC line
        ax.plot(self.positions, self.gc_values, 'b-', linewidth=1, label='GC Content')
        
        # Fill areas
        ax.fill_between(self.positions, self.gc_values, 50,
                       where=(self.gc_values >= 50), alpha=0.3, color='blue')
        ax.fill_between(self.positions, self.gc_values, 50,
                       where=(self.gc_values < 50), alpha=0.3, color='red')
        
        # Reference lines
        ax.axhline(y=50, color='black', linestyle='--', linewidth=0.5, label='50% GC')
        
        # Highlight GC islands
        if highlight_islands:
            islands = self.find_gc_islands(threshold=threshold)
            for island in islands:
                ax.axvspan(island['start'], island['end'], 
                          alpha=0.2, color='green')
            ax.axhline(y=threshold, color='green', linestyle=':', 
                      linewidth=1, label=f'Island threshold ({threshold}%)')
        
        ax.set_xlabel('Position (bp)', fontsize=12)
        ax.set_ylabel('GC Content (%)', fontsize=12)
        ax.set_title(f'GC Content Landscape (window={self.window_size}, step={self.step})', fontsize=14)
        ax.set_ylim(0, 100)
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig
    
    def plot_distribution(self):
        """Plot GC content distribution."""
        if self.gc_values is None:
            self.calculate()
        
        fig, ax = plt.subplots(figsize=(10, 5))
        
        ax.hist(self.gc_values, bins=50, edgecolor='white', color='steelblue', density=True)
        ax.axvline(np.mean(self.gc_values), color='red', linestyle='--',
                  label=f'Mean: {np.mean(self.gc_values):.1f}%')
        ax.axvline(50, color='black', linestyle=':',
                  label='50% reference')
        
        ax.set_xlabel('GC Content (%)', fontsize=12)
        ax.set_ylabel('Density', fontsize=12)
        ax.set_title('Distribution of Window GC Content', fontsize=14)
        ax.legend()
        
        plt.tight_layout()
        return fig


# Test with synthetic sequence having variable GC
np.random.seed(42)
seq_parts = []
for i in range(20):
    # Alternate between AT-rich and GC-rich regions
    if i % 2 == 0:
        weights = [0.35, 0.15, 0.15, 0.35]  # AT-rich
    else:
        weights = [0.15, 0.35, 0.35, 0.15]  # GC-rich
    seq_parts.append(''.join(np.random.choice(['A', 'C', 'G', 'T'], 
                                              size=500, p=weights)))
test_sequence = ''.join(seq_parts)

gc_analyzer = GCLandscape(test_sequence, window_size=100, step=20)
gc_analyzer.calculate()

print("Statistics:")
for key, value in gc_analyzer.statistics().items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

print(f"\nGC Islands found: {len(gc_analyzer.find_gc_islands())}")

gc_analyzer.plot()
plt.show()

gc_analyzer.plot_distribution()
plt.show()

## Part 4: Genome Analyzer Pipeline

In [ ]:
class GenomeAnalyzer:
    """Complete genome analysis pipeline."""
    
    def __init__(self, sequence=None, fasta_file=None):
        """Initialize with sequence or FASTA file."""
        if fasta_file:
            record = next(SeqIO.parse(fasta_file, 'fasta'))
            self.sequence = str(record.seq).upper()
            self.name = record.id
        elif sequence:
            self.sequence = sequence.upper()
            self.name = "Unknown"
        else:
            raise ValueError("Must provide sequence or fasta_file")
        
        # Initialize analyzers
        self.orf_finder = None
        self.gc_analyzer = None
        self.results = {}
    
    def analyze_composition(self):
        """Analyze nucleotide composition."""
        total = len(self.sequence)
        composition = {
            'A': self.sequence.count('A'),
            'T': self.sequence.count('T'),
            'G': self.sequence.count('G'),
            'C': self.sequence.count('C'),
            'N': self.sequence.count('N')
        }
        
        composition['gc_content'] = (composition['G'] + composition['C']) / total * 100
        composition['at_content'] = (composition['A'] + composition['T']) / total * 100
        composition['total_length'] = total
        
        self.results['composition'] = composition
        return composition
    
    def find_orfs(self, min_length=100):
        """Find all ORFs."""
        self.orf_finder = ORFFinder(self.sequence, min_length=min_length)
        self.orf_finder.find_all_orfs()
        self.results['orfs'] = self.orf_finder.summary()
        return self.orf_finder.orfs
    
    def analyze_gc_landscape(self, window_size=100, step=20):
        """Analyze GC content landscape."""
        self.gc_analyzer = GCLandscape(self.sequence, window_size, step)
        self.gc_analyzer.calculate()
        self.results['gc_landscape'] = self.gc_analyzer.statistics()
        return self.gc_analyzer
    
    def analyze_kmers(self, k=4):
        """Analyze k-mer frequencies."""
        kmers = [self.sequence[i:i+k] for i in range(len(self.sequence) - k + 1)]
        kmer_counts = Counter(kmers)
        
        total_kmers = len(kmers)
        unique_kmers = len(kmer_counts)
        possible_kmers = 4 ** k
        
        self.results['kmers'] = {
            'k': k,
            'total_kmers': total_kmers,
            'unique_kmers': unique_kmers,
            'possible_kmers': possible_kmers,
            'coverage': unique_kmers / possible_kmers * 100,
            'most_common': kmer_counts.most_common(10),
            'least_common': kmer_counts.most_common()[-10:]
        }
        return self.results['kmers']
    
    def full_analysis(self, min_orf_length=100, gc_window=100, gc_step=20, kmer_k=4):
        """Run complete analysis pipeline."""
        print(f"Analyzing sequence: {self.name}")
        print(f"Sequence length: {len(self.sequence):,} bp")
        print("-" * 50)
        
        # Composition
        print("\n1. Nucleotide Composition:")
        comp = self.analyze_composition()
        print(f"   GC Content: {comp['gc_content']:.2f}%")
        print(f"   AT Content: {comp['at_content']:.2f}%")
        
        # ORFs
        print(f"\n2. ORF Analysis (min length = {min_orf_length}):")
        self.find_orfs(min_length=min_orf_length)
        orf_stats = self.results['orfs']
        print(f"   Total ORFs: {orf_stats['total_orfs']}")
        if orf_stats['total_orfs'] > 0:
            print(f"   Forward strand: {orf_stats['forward_strand']}")
            print(f"   Reverse strand: {orf_stats['reverse_strand']}")
            print(f"   Mean length: {orf_stats['mean_length']:.0f} bp")
        
        # GC landscape
        print(f"\n3. GC Landscape (window = {gc_window}, step = {gc_step}):")
        self.analyze_gc_landscape(window_size=gc_window, step=gc_step)
        gc_stats = self.results['gc_landscape']
        print(f"   Mean: {gc_stats['mean_window_gc']:.2f}%")
        print(f"   Std Dev: {gc_stats['std_window_gc']:.2f}%")
        print(f"   Range: {gc_stats['min_gc']:.1f}% - {gc_stats['max_gc']:.1f}%")
        
        islands = self.gc_analyzer.find_gc_islands()
        print(f"   GC Islands found: {len(islands)}")
        
        # K-mers
        print(f"\n4. {kmer_k}-mer Analysis:")
        kmer_stats = self.analyze_kmers(k=kmer_k)
        print(f"   Unique {kmer_k}-mers: {kmer_stats['unique_kmers']}")
        print(f"   Coverage: {kmer_stats['coverage']:.1f}%")
        print(f"   Most common: {kmer_stats['most_common'][:5]}")
        
        print("\n" + "=" * 50)
        print("Analysis complete!")
        
        return self.results
    
    def generate_report(self, output_file=None):
        """Generate analysis report."""
        report = []
        report.append(f"# Genome Analysis Report: {self.name}")
        report.append(f"\n## Sequence Overview")
        report.append(f"- Length: {len(self.sequence):,} bp")
        
        if 'composition' in self.results:
            comp = self.results['composition']
            report.append(f"\n## Nucleotide Composition")
            report.append(f"- A: {comp['A']:,} ({comp['A']/comp['total_length']*100:.1f}%)")
            report.append(f"- T: {comp['T']:,} ({comp['T']/comp['total_length']*100:.1f}%)")
            report.append(f"- G: {comp['G']:,} ({comp['G']/comp['total_length']*100:.1f}%)")
            report.append(f"- C: {comp['C']:,} ({comp['C']/comp['total_length']*100:.1f}%)")
            report.append(f"- GC Content: {comp['gc_content']:.2f}%")
        
        if 'orfs' in self.results:
            orf_stats = self.results['orfs']
            report.append(f"\n## ORF Analysis")
            report.append(f"- Total ORFs: {orf_stats['total_orfs']}")
            if orf_stats['total_orfs'] > 0:
                report.append(f"- Forward strand: {orf_stats['forward_strand']}")
                report.append(f"- Reverse strand: {orf_stats['reverse_strand']}")
                report.append(f"- Length range: {orf_stats['min_length']} - {orf_stats['max_length']} bp")
        
        report_text = '\n'.join(report)
        
        if output_file:
            with open(output_file, 'w') as f:
                f.write(report_text)
        
        return report_text


# Test
analyzer = GenomeAnalyzer(sequence=test_sequence)
results = analyzer.full_analysis(min_orf_length=90, gc_window=100, gc_step=20, kmer_k=4)

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. GC Landscape
ax = axes[0, 0]
ax.plot(analyzer.gc_analyzer.positions, analyzer.gc_analyzer.gc_values, 'b-', linewidth=1)
ax.axhline(y=50, color='red', linestyle='--', linewidth=0.5)
ax.fill_between(analyzer.gc_analyzer.positions, analyzer.gc_analyzer.gc_values, 50,
               where=(analyzer.gc_analyzer.gc_values >= 50), alpha=0.3, color='blue')
ax.fill_between(analyzer.gc_analyzer.positions, analyzer.gc_analyzer.gc_values, 50,
               where=(analyzer.gc_analyzer.gc_values < 50), alpha=0.3, color='red')
ax.set_xlabel('Position')
ax.set_ylabel('GC %')
ax.set_title('GC Content Landscape')

# 2. ORF Length Distribution
ax = axes[0, 1]
if analyzer.orf_finder.orfs:
    lengths = [orf['length'] for orf in analyzer.orf_finder.orfs]
    ax.hist(lengths, bins=20, edgecolor='white', color='steelblue')
    ax.axvline(np.mean(lengths), color='red', linestyle='--')
ax.set_xlabel('ORF Length (bp)')
ax.set_ylabel('Count')
ax.set_title('ORF Length Distribution')

# 3. GC Distribution
ax = axes[1, 0]
ax.hist(analyzer.gc_analyzer.gc_values, bins=30, edgecolor='white', color='steelblue')
ax.axvline(np.mean(analyzer.gc_analyzer.gc_values), color='red', linestyle='--')
ax.set_xlabel('GC Content (%)')
ax.set_ylabel('Count')
ax.set_title('GC Content Distribution')

# 4. Nucleotide Composition
ax = axes[1, 1]
comp = analyzer.results['composition']
nucs = ['A', 'T', 'G', 'C']
counts = [comp[n] for n in nucs]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
ax.bar(nucs, counts, color=colors)
ax.set_xlabel('Nucleotide')
ax.set_ylabel('Count')
ax.set_title('Nucleotide Composition')

plt.tight_layout()
plt.show()

---

## Summary

Key BioPython and analysis techniques:

1. **BioPython Basics**: SeqIO, Seq, SeqRecord, gc_fraction
2. **ORF Finding**: 6-frame translation, start/stop codon detection
3. **GC Landscape**: Sliding window analysis, CpG island detection
4. **Visualization**: Matplotlib integration for scientific plots
5. **OOP Design**: Reusable analysis classes with methods